In [ ]:
import os
import json
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from rapidfuzz import fuzz, process
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import SparsePCA
from sklearn.model_selection import train_test_split

### Load Data

In [ ]:
main_directory = os.getenv("POND_PATH")
data_directory = os.getenv("POND_DATA_PATH")
text_directory = os.getenv("POND_TEXT_PATH")

In [ ]:
# Directory
with open(os.path.join(main_directory, "directory.json"), "r") as f:
    paper_info = json.load(f)
registered_titles = [entry['title'] for entry in paper_info.values()]
registered_titles.sort()
registered_titles = registered_titles[:10]

# Original dataset
pond_data = pd.read_csv(os.path.join(data_directory, "pond_data.csv"), encoding_errors='ignore')
pond_data = pond_data.loc[pond_data.title.isin(registered_titles)]
pond_df = pond_data.loc[:,['author', 'title', 'pondname', 'location', 'author_term',
            'max_depth_m', 'mean_surfacearea_m2', 'macrophytes_percentcover', 'ph', 'tn_ugpl', 'tp_ugpl', 'chla_ugpl']]
pond_df.columns = ['author', 'title', 'name', 'location', 'ecosystem',
            'max_depth', 'surface_area', 'vegetation_cover', 'ph', 'tn', 'tp', 'chla']

# Split the dataframe's rows so that each measurement is in its own row
pond_df = pond_df.melt(id_vars=['author', 'title', 'name', 'location', 'ecosystem'], 
                       value_vars=['max_depth', 'surface_area', 'vegetation_cover', 'ph', 'tn', 'tp', 'chla'],
                       var_name='measurement', value_name='value')
pond_df = pond_df.dropna(subset=['value'])
pond_df = pond_df.reset_index(drop=True)
n_entries = pond_df.shape[0]

In [ ]:
def process(
    result_df,
    conversion_table
):
    result_df = result_df.copy()

    # Drop rows without any measurements
    result_df = result_df.dropna(subset=['value'])
    result_df = result_df.reset_index(drop=True)

    # Convert units
    for row_idx, row in result_df.iterrows():
        measurement_type = row['measurement']
        val = row['value']
        unit = row['units']
        if conversion_table.get(measurement_type) is not None:
            if conversion_table[measurement_type].get(unit) is not None:
                conversion_factor = conversion_table[measurement_type][unit]
                result_df.at[row_idx, 'value'] = val * conversion_factor
            else:
                result_df.at[row_idx, 'value'] = np.nan # Units not recognized

    result_df = result_df.dropna(subset=['value'])
    result_df = result_df.reset_index(drop=True)

    # Drop all unit columns
    result_df = result_df.drop(columns=["units"])

    # Drop exact duplicates
    result_df = result_df.drop_duplicates()

    # Reset index
    result_df = result_df.reset_index(drop=True)

    return result_df


conversion_table = {
    'depth': {"cm": 0.01, "feet": 0.3048, "km": 1000, "m": 1},
    'surface_area': {"km^2": 1e6, "ha": 1e4, "mi^2": 2.59e6, "m^2": 1},
    'vegetation_cover': {"percent": 0.01, "fraction": 1},
    'tn': {"mg/L": 1000, "µg/L": 1, "μmol/L": 14.01, "ppm": 1000, "ppb": 1},
    'tp': {"mg/L": 1000, "µg/L": 1, "μmol/L": 30.97, "ppm": 1000, "ppb": 1},
    'chl': {"mg/L": 1000, "µg/L": 1}
}

In [ ]:
ignore_measurements = ['latitude', 'longitude'] # Ignoring these for now because they are not in the original dataset

result_df = pd.read_csv('../data/pond_results_10_papers_better_ocr.csv')
result_df = result_df.loc[~result_df.measurement.isin(ignore_measurements)]
result_df['value'] = result_df['value'].str.replace(',', '')  # Remove commas from numbers
result_df['value'] = pd.to_numeric(result_df['value'], errors='coerce')
result_df = process(result_df, conversion_table)
result_df.sort_values(by=["title"], inplace=True)
result_df = result_df.reset_index(drop=True)

result_df_validated = pd.read_csv('../data/pond_results4_validated.csv')
result_df_validated = result_df_validated.loc[~result_df_validated.measurement.isin(ignore_measurements)]
result_df_validated['value'] = result_df_validated['value'].str.replace(',', '')  # Remove commas from numbers
result_df_validated['value'] = pd.to_numeric(result_df_validated['value'], errors='coerce')
result_df_validated = process(result_df_validated, conversion_table)

In [ ]:
pond_df.loc[pond_df.title == registered_titles[7], :].sort_values(by='measurement')

In [ ]:
result_df.loc[result_df.title == registered_titles[7], :].iloc[:, :12].sort_values(by='measurement')

In [ ]:
print(result_df.loc[result_df.title == registered_titles[7], :].iloc[:, :11].sort_values(by='measurement').iloc[-1,:].context)

In [ ]:
result_df.loc[result_df.title == registered_titles[7], :].iloc[:, :11]

In [ ]:
print(result_df.loc[result_df.title == registered_titles[7], :].iloc[0,:].context)

In [ ]:
filepath = os.path.join(text_directory, "biodiversity_of_constructed.json")
chunks = []
with open(filepath, 'r', encoding='utf-8') as file:
    doc_chunks = json.load(file)
    chunks.append(doc_chunks)

In [ ]:
print(chunks[0][23])

### LLM as Judge Baseline

In [ ]:
status_dict = {
    'valid': 1,
    'hallucination': 0,
    'disorientation': 0,
    'deviation': 0,
}

y_judge = result_df_validated['judgement']
y_valid = result_df_validated['validation_status'].map(status_dict)

accuracy = (y_judge == y_valid).mean()
print(f"LLM Judge Accuracy: {accuracy:.4f}")

### Manually validated model

In [ ]:
context_columns = []
context_columns_by_layer = [[] for _ in range(32)]
parametric_columns = []

for col in result_df_validated.columns:
    if col.startswith('context_'):
        context_columns.append(col)
        layer_num = int(col.split('_')[-1].split('h')[0][1:])
        context_columns_by_layer[layer_num].append(col)
    elif col.startswith('parametric_'):
        parametric_columns.append(col)

In [ ]:
pond_results_aggregate = result_df_validated.drop(columns = context_columns)
context_columns_aggregate = []
for layer_num in range(32):
    cols = context_columns_by_layer[layer_num]
    pond_results_aggregate[f'context_l{layer_num}'] = result_df_validated[cols].mean(axis=1)
    context_columns_aggregate.append(f'context_l{layer_num}')

In [ ]:
status_dict = {
    'valid': 1,
    'hallucination': 0,
    'disorientation': 0,
    'deviation': 1,
}

X = result_df_validated[context_columns + parametric_columns]
y = result_df_validated['validation_status'].map(status_dict)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Fitting regularization parameter for logistic regression
regularization_params = np.linspace(0.0, 0.001, 101)
scores = np.empty((len(regularization_params), 5))

for i, alpha in enumerate(regularization_params):
    model = LogisticRegression(C=1/(alpha + 1e-10), penalty = "l1", solver = "liblinear", max_iter=1000)
    cv_results = cross_validate(model, X, y, cv=5, scoring='accuracy')
    scores[i, :] = cv_results['test_score']

In [ ]:
np.mean(scores, axis=1)

In [ ]:
regularization_params[10]

In [ ]:
plt.plot(regularization_params, np.mean(scores, axis = 1))
plt.xlabel('Regularization Parameter (alpha)')
plt.ylabel('Reconstruction Error')

In [ ]:
regularization_params[np.argmax(np.mean(scores, axis=1))]

In [ ]:
# Fit logistic regression model
alpha = 9e-05
model = LogisticRegression(C = 1/(alpha + 1e-10), penalty = "l1", solver='liblinear', max_iter=1000, random_state = 342)
model.fit(X, y)
#train_accuracy = model.score(X_train, y_train)
#print(f"Logistic Regression Train Accuracy: {train_accuracy:.4f}")
#accuracy = model.score(X, y)
#print(f"Logistic Regression Test Accuracy: {accuracy:.4f}")

coefficients = model.coef_[0]
context_coefficients = coefficients[:-32].reshape(32, 32)
parametric_coefficients = coefficients[-32:]

In [ ]:
model.predict(X_test)

In [ ]:
y_test

In [ ]:
pond_df

In [ ]:
fig,ax = plt.subplots(figsize=(6,6), dpi = 100)
pos = ax.matshow(context_coefficients, cmap='BuPu')
ax.set_ylabel('Layers')
ax.set_xlabel('Heads')
ax.set_title(f'Context Score Coefficients')
fig.colorbar(pos, ax=ax)

In [ ]:
fig,ax = plt.subplots(figsize=(6,4), dpi = 100)
ax.bar(
    range(len(parametric_coefficients)), parametric_coefficients, alpha = 0.5
)
ax.set_ylabel('Parametric Score Coefficients')
ax.set_xlabel('Layers')

### Matching to dataset

In [ ]:
'''
def match_datapoints(ground_truth, extracted):
    matching = {}
    for i, row in ground_truth.iterrows():
        title = row['title']
        name = row['name']
        measurement = row['measurement']
        val = row['value']

        if title in extracted['title'].values:
            extracted_title_entries = extracted.loc[extracted['title'] == title]
            
            # find any close matches in name
            extracted_title_names = extracted_title_entries['name'].values
            indx = []
            for j, name in enumerate(extracted_title_names):
                ratio = fuzz.ratio(row['name'], name)
                if ratio > 0:  # threshold for a close match
                    indx.append(extracted_title_entries.index[j])

            if len(indx) > 0:
                extracted_title_entries = extracted_title_entries.loc[indx]
                extracted_title_values = extracted_title_entries.loc[extracted_title_entries.measurement == measurement]['value']

                # check if any value is close enough
                close_enough = np.isclose(extracted_title_values, val, atol=1e-3)
                if np.any(close_enough):
                    if np.sum(close_enough) > 1:
                        print(f"Warning: multiple close matches found for row {i} in ground truth.")
                    closest = np.where(close_enough)[0][0]
                    extracted_row = extracted_title_entries.iloc[closest]
                    matching[i] = extracted_row.name

    return matching
'''

def match_datapoints(ground_truth, extracted):
    edges = []
    edge_weights = []
    for i, row_gt in ground_truth.iterrows():
        for j, row_ex in extracted.iterrows():
            if (
                row_gt['title'] == row_ex['title'] and 
                row_gt['measurement'] == row_ex['measurement'] and 
                np.isclose(row_gt['value'], row_ex['value'], atol=1e-3)
            ):
                name_similarity = fuzz.ratio(row_gt['name'], row_ex['name']) / 100.0
                location_similarity = fuzz.ratio(row_gt['location'], row_ex['location']) / 100.0
                ecosystem_similarity = fuzz.ratio(row_gt['ecosystem'], row_ex['ecosystem']) / 100.0
                weight = (name_similarity + location_similarity + ecosystem_similarity) / 3.0
                edges.append((i, j))
                edge_weights.append(weight)

    print(f"Total edges found: {len(edges)}")
    # Create a bipartite graph and find maximum weight matching
    G = nx.Graph()
    G.add_edges_from([(f"gt_{i}", f"ex_{j}", {'weight': w}) for (i, j), w in zip(edges, edge_weights)])
    matching = nx.algorithms.matching.max_weight_matching(G)
    index_matching = []
    for i, j in matching:
        if i.startswith("gt_"):
            index_matching.append((int(i[3:]), int(j[3:])))
        else:
            index_matching.append((int(j[3:]), int(i[3:])))
    
    return index_matching


def estimate_precision_recall(ground_truth, extracted, measurement_cols):
    total_ground_truth = ground_truth.shape[0]
    total_extracted = extracted.shape[0]

    matched_indices = set()
    for col in measurement_cols:
        matches = match_datapoints(ground_truth, extracted, col)
        matched_indices.update(matches.values())

    true_positives = len(matched_indices)
    precision = true_positives / total_extracted if total_extracted > 0 else 0
    recall = true_positives / total_ground_truth if total_ground_truth > 0 else 0

    return precision, recall

In [ ]:
matching = match_datapoints(pond_df, result_df)

In [ ]:
len(matching)

In [ ]:
matching

In [ ]:
match = 12
print("Ground Truth:")
print(pond_df.iloc[matching[match][0], :])
print("\nExtracted:")
print(result_df.iloc[matching[match][1], :11])

In [ ]:
20 * 12